# 08 Image Level Evaluation

In [ ]:
!rm -rf /content/unet-ensemble

In [ ]:
!git clone https://github.com/JeserylMae/unet-ensemble.git

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import sys
sys.path.append('/content/unet-ensemble')

In [ ]:
!pip install safetensors huggingface_hub segmentation-models-pytorch scikit-image scipy -q

In [ ]:
import os
import torch
import numpy as np
import pandas as pd

from tqdm import tqdm
from huggingface_hub import login
from torch.utils.data import DataLoader

from src.dataset.dataset    import Dataset as LazyDS
from src.dataset.dataloader import DataLoader as DSLoader
from src.eval.evaluate      import Evaluate
from src.eval.metrics       import Metrics
from src.dataset.rgb_dataset import RGBDataset, RGBDataLoader

In [ ]:
token = '' 
login(token=token)

## 08-1 Configuration

In [ ]:
# Dataset 
IMG_SIZE     = 256
DATASET_ROOT = '/content/drive/Shareddrives/U-Net Ensemble/Dataset'
BATCH_SIZE   = 1
NUM_WORKERS  = 2

MASK_FOLDER         = 'GTA - Masks'
PRNU_FOLDER         = 'Feature - PRNU'
ILLUMINATION_FOLDER = 'Feature - Illumination'
FREQUENCY_FOLDER    = 'Feature - Frequency'
CATEGORIES          = ['Synthetic', 'Authentic']
TEMPLATES           = [
    'template-a', 'template-albania', 'template-b',
    'template-c', 'template-chile',   'template-deutschland',
    'template-spain', 'template-usa',
]

# Feature combination 
# Must match the combination used when training the checkpoint.
# Options: ('prnu','illumination','frequency') | ('prnu','frequency')
#          ('prnu','illumination')             | ('frequency','illumination')
FEATURES = ('frequency', 'illumination')

# HuggingFace repos 
HF_USERNAME  = 'Jesayyy'
MODEL_REPO  = f'{HF_USERNAME}/mben-baselines'
SUBFOLDER    = 'frequency_illumination'   # change to match the checkpoint subfolder

# Output CSV path (Google Drive) 
CSV_DIR  = '/content/drive/Shareddrives/U-Net Ensemble/checkpoint/U-Net Ensemble'
CSV_PATH = os.path.join(CSV_DIR, 'per_image_evaluation_results-frequency_illumination.csv')

# Misc 
THRESHOLD      = 0.5
BOUNDARY_WIDTH = 2       # dilation tolerance for BF Score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device  : {device}')
print(f'Features: {sorted(FEATURES)}')

## 08-2 U-Net++ / Attention U-Net

### A. Dataset Preparation

In [ ]:
ds_loader = DSLoader(
    mask_folder         = MASK_FOLDER,
    prnu_folder         = PRNU_FOLDER,
    illumination_folder = ILLUMINATION_FOLDER,
    frequency_folder    = FREQUENCY_FOLDER,
    categories          = CATEGORIES,
    templates           = TEMPLATES,
    features            = FEATURES,
)

eval_samples = ds_loader.load_images('Evaluation', DATASET_ROOT)
eval_ds = LazyDS(eval_samples, IMG_SIZE, augment=False, features=FEATURES)

# batch_size=1 is required so each row in the CSV corresponds to exactly one image
eval_loader = DataLoader(
    eval_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
)

print(f'Features     : {sorted(FEATURES)}')
print(f'Total images : {len(eval_samples)}')
print(f'Total batches: {len(eval_loader)}')

### B. Load Model

In [ ]:
evaluator = Evaluate(device=device, features=FEATURES, threshold=THRESHOLD,
                     boundary_width=BOUNDARY_WIDTH)

model  = evaluator.load_attentionunet_from_hub(repo_id=MODEL_REPO, subfolder=SUBFOLDER)

print('Both models loaded and ready.')

### C. Inference Loop

In [ ]:
FEATURE_ORDER   = ('prnu', 'illumination', 'frequency')
active_features = [f for f in FEATURE_ORDER if f in frozenset(FEATURES)]

rows = []

model.eval()

with torch.no_grad():
    for img_idx, batch in enumerate(tqdm(eval_loader, desc='Evaluating per image')):

        # Unpack batch 
        # batch = (*feature_tensors_in_canonical_order, fused_t, mask_t)
        *feat_tensors, fused, masks = batch

        feature_dict = {
            feat: feat_tensors[i].to(device)
            for i, feat in enumerate(active_features)
        }
        fused = fused.to(device)
        masks = masks.to(device)

        # Grab the source file path for this sample (for the CSV) 
        sample_path = eval_samples[img_idx].get('mask', f'image_{img_idx:05d}')
        image_name  = os.path.basename(sample_path)

        gt_np = masks.cpu().numpy()[0, 0].astype(np.uint8) 

        logits_unetpp = model(feature_dict, fused)   # (1, 1, H, W)
        prob_unetpp   = torch.sigmoid(logits_unetpp).cpu().numpy()[0, 0]  # (H, W)
        pred_unetpp   = (prob_unetpp >= THRESHOLD).astype(np.uint8)

        m = Metrics()
        metrics = m.compute_all_metrics(prob_unetpp, pred_unetpp, gt_np, BOUNDARY_WIDTH)

        # Collect row 
        row = {
            'image_index' : img_idx,
            'image_name'  : image_name,
            'features'    : '+'.join(sorted(FEATURES)),
            'Feature_Correlation': metrics['Feature_Correlation'],
            'Dice'               : metrics['Dice'],
            'IoU'                : metrics['IoU'],
            'Pixel_Accuracy'     : metrics['Pixel_Accuracy'],
            'BF_Score'           : metrics['BF_Score']
        }
        rows.append(row)

print(f'\nInference complete. {len(rows)} images evaluated.')

## 08-3 Baseline

### A. Dataset Preparation

In [ ]:
eval_loader = RGBDataLoader(
    mask_folder = MASK_FOLDER,
    rgb_folder  = "Normalized",
    categories  = CATEGORIES,
    templates   = TEMPLATES
    )

eval_samples = eval_loader.load_images('Evaluation', DATASET_ROOT)

eval_ds = RGBDataset(eval_samples, IMG_SIZE, augment=True)

rgb_loader = DataLoader(
    eval_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
)

print(f'Total images : {len(eval_samples)}')
# print(f'Total batches: {len(eval_loader)}')

### B. Load Model

In [ ]:
evaluator = Evaluate(device=device, features=FEATURES)
model = evaluator.load_rgb_baseline_from_hub(MODEL_REPO, SUBFOLDER)

print('Baseline model loaded and ready.')

### C. Inference Loop

In [ ]:
baseline_rows = []

model.eval()

with torch.no_grad():
    for img_idx, batch in enumerate(tqdm(rgb_loader, desc='Baseline — evaluating per image')):

        # RGBDataset yields (rgb_t, mask_t) — only 2 tensors
        # Do NOT unpack as *feat_tensors, fused, masks
        rgb, masks = batch
        rgb   = rgb.to(device)    # (1, 3, H, W)
        masks = masks.to(device)  # (1, 1, H, W)

        sample_path = eval_samples[img_idx].get('mask', f'image_{img_idx:05d}')
        image_name  = os.path.basename(sample_path)

        gt_np = masks.cpu().numpy()[0, 0].astype(np.uint8)   # (H, W)

        # RGB baseline forward: model(rgb), NOT model(feature_dict, fused)
        prob = torch.sigmoid(model(rgb)).cpu().numpy()[0, 0]
        pred = (prob >= THRESHOLD).astype(np.uint8)

        m = Metrics().compute_all_metrics(prob, pred, gt_np, BOUNDARY_WIDTH)

        baseline_rows.append({
            'image_index'        : img_idx,
            'image_name'         : image_name,
            'Feature_Correlation': m['Feature_Correlation'],
            'Dice'               : m['Dice'],
            'IoU'                : m['IoU'],
            'Pixel_Accuracy'     : m['Pixel_Accuracy'],
            'BF_Score'           : m['BF_Score'],
        })

print(f'\nInference complete. {len(baseline_rows)} images evaluated.')

## 08-4 Unet Ensemble

### A. Configuration

In [ ]:
UNETPP_REPO  = f'{HF_USERNAME}/mben-unetpp'
ATTUNET_REPO = f'{HF_USERNAME}/mben-attentionunet'
SUBFOLDER    = 'frequency_illumination' 

ALPHA        = 0.5

FEATURE_TAG  = '+'.join(sorted(FEATURES))

### B. Dataset Preparation

In [ ]:
ds_loader = DSLoader(
    mask_folder         = MASK_FOLDER,
    prnu_folder         = PRNU_FOLDER,
    illumination_folder = ILLUMINATION_FOLDER,
    frequency_folder    = FREQUENCY_FOLDER,
    categories          = CATEGORIES,
    templates           = TEMPLATES,
    features            = FEATURES,
)

eval_samples = ds_loader.load_images('Evaluation', DATASET_ROOT)
eval_ds      = LazyDS(eval_samples, IMG_SIZE, augment=False, features=FEATURES)

# batch_size=1 is required so each row in the CSV corresponds to exactly one image
eval_loader = DataLoader(
    eval_ds,
    batch_size  = BATCH_SIZE,
    shuffle     = False,
    num_workers = NUM_WORKERS,
    pin_memory  = True,
)

print(f'Features     : {sorted(FEATURES)}')
print(f'Total images : {len(eval_samples)}')
print(f'Total batches: {len(eval_loader)}')

### C. Load Models

In [ ]:
evaluator = Evaluate(device=device, features=FEATURES, threshold=THRESHOLD,
                     boundary_width=BOUNDARY_WIDTH)

model_unetpp  = evaluator.load_unetpp_from_hub(UNETPP_REPO,  SUBFOLDER)
model_attunet = evaluator.load_attentionunet_from_hub(ATTUNET_REPO, SUBFOLDER)

model_unetpp.eval()
model_attunet.eval()

print('U-Net++ and Attention U-Net loaded and ready.')

### D. Inference Loop

In [ ]:
FEATURE_ORDER   = ('prnu', 'illumination', 'frequency')
active_features = [f for f in FEATURE_ORDER if f in frozenset(FEATURES)]

rows = []

with torch.no_grad():
    for img_idx, batch in enumerate(tqdm(eval_loader, desc='Ensemble — evaluating per image')):

        # Unpack batch
        # batch = (*feature_tensors_in_canonical_order, fused_t, mask_t)
        *feat_tensors, fused, masks = batch

        feature_dict = {
            feat: feat_tensors[i].to(device)
            for i, feat in enumerate(active_features)
        }
        fused = fused.to(device)
        masks = masks.to(device)

        sample_path = eval_samples[img_idx].get('mask', f'image_{img_idx:05d}')
        image_name  = os.path.basename(sample_path)

        gt_np = masks.cpu().numpy()[0, 0].astype(np.uint8)   # (H, W)

        # ── Individual decoder forward passes ──────────────────────────────────
        logits_unetpp  = model_unetpp(feature_dict, fused)    # (1, 1, H, W)
        logits_attunet = model_attunet(feature_dict, fused)   # (1, 1, H, W)

        prob_unetpp  = torch.sigmoid(logits_unetpp).cpu().numpy()[0, 0]   # (H, W)
        prob_attunet = torch.sigmoid(logits_attunet).cpu().numpy()[0, 0]  # (H, W)

        # ── Weighted ensemble fusion ────────────────────────────────────────────
        prob_ensemble = ALPHA * prob_unetpp + (1.0 - ALPHA) * prob_attunet  # (H, W)
        pred_ensemble = (prob_ensemble >= THRESHOLD).astype(np.uint8)        # (H, W)

        # ── Per-image metrics on the fused mask ────────────────────────────────
        m       = Metrics()
        metrics = m.compute_all_metrics(prob_ensemble, pred_ensemble, gt_np, BOUNDARY_WIDTH)

        rows.append({
            'image_index'        : img_idx,
            'image_name'         : image_name,
            'features'           : FEATURE_TAG,
            'alpha'              : ALPHA,
            'Feature_Correlation': metrics['Feature_Correlation'],
            'Dice'               : metrics['Dice'],
            'IoU'                : metrics['IoU'],
            'Pixel_Accuracy'     : metrics['Pixel_Accuracy'],
            'BF_Score'           : metrics['BF_Score'],
        })

print(f'\nInference complete. {len(rows)} images evaluated.')

## 08-4 Build Dataframe & Preview

In [ ]:
results_df = pd.DataFrame(rows)

pd.set_option('display.float_format', '{:.6f}'.format)
pd.set_option('display.max_columns', None)

print(f'Shape: {results_df.shape}')
results_df.head(10)

## 08-5 Save CSV to Drive

In [ ]:
os.makedirs(CSV_DIR, exist_ok=True)

if os.path.exists(CSV_PATH):
    # Append without re-writing the header
    results_df.to_csv(CSV_PATH, mode='a', header=False, index=False)
    print(f'Results appended to existing file: {CSV_PATH}')
else:
    results_df.to_csv(CSV_PATH, mode='w', header=True, index=False)
    print(f'Results saved to new file: {CSV_PATH}')

print(f'Total rows written: {len(results_df)}')
print(f'Total columns     : {len(results_df.columns)}')